# Thanglish Agent: dataset pipeline

Builds, verifies and packages the **Thanglish -> Tamil** fine-tuning dataset. Run it from the repo root (or `cd` there first).

Real Thanglish text is scarce, so the dataset is built backwards: real Tamil sentences from `data/` are transliterated by a deterministic rule engine (`thanglish_transliterator.py`) to make the Thanglish side, then flipped so **Thanglish is the input and the original Tamil is the target**.

**Scope.** This notebook covers the data pipeline only. It does not download or run a model, train LoRA adapters, or implement an agent. Those parts are not in this repo.

In [ ]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd()
assert (ROOT / "thanglish_transliterator.py").exists(), "Run this notebook from the repo root"

def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True, cwd=ROOT)

print("Project:", ROOT)

## 1. Install the dependency

Only `open-tamil` is needed (it provides the Tamil letter splitter).

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)

## 2. Build the dataset

Reads the 72 Tamil files in `data/` (alphabetical order), keeps lines of 8-300 characters that contain Tamil and have not been seen before, stops at 903, transliterates each, and writes `train.jsonl` plus 903 single-record files in `output/`. The result is deterministic, so this reproduces the committed files.

In [ ]:
run("convert_data_to_thanglish_jsonl.py")

## 3. Verify

Checks record count, schema, uniqueness and length statistics, then (`--regen`) re-runs the transliterator on every Tamil target and confirms it reproduces the stored Thanglish input.

In [ ]:
run("verify_dataset.py", "--regen")

## 4. Package

Requires exactly 903 valid files, writes a manifest with SHA-256 hashes, builds a ZIP beside `output/`, and verifies its CRCs before renaming it into place.

In [ ]:
run("package_thanglish_files.py")

## 5. Look at a few pairs

In [ ]:
import json
with open(ROOT / "train.jsonl", encoding="utf-8") as f:
    for i, line in zip(range(3), f):
        r = json.loads(line)
        print("Thanglish (input):", r["prompt"][1]["content"])
        print("Tamil (target):   ", r["completion"][0]["content"])
        print()

## Not covered here

Everything below needs files that are **not in this repo**, so it is intentionally left out:

- downloading the 18.8 GB base checkpoint (`tl_clm_model.zip`) and the SentencePiece tokenizer (`tamil_sp_v4.model`)
- LoRA training and inference (`train_lora.py`, `infer.py`)
- the 60-sample transliteration evaluation (`evaluate_rules.py`) and automated tests

The transliteration is lossy (ந/ன/ண all become `n`, ல/ள become `l`, ர/ற become `r`), which is exactly what a model trained on this data would have to resolve from context.